In [1]:
##imports##
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
##Directories##
OOT = Path.cwd()

DATA_DIR = ROOT / "data"

SYNTHETIC_DIR = DATA_DIR / "synthetic"

PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = ROOT / "outputs"

FIGURE_DIR = ROOT / "figures"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [20]:
##load Dataset##
dataset = pd.read_csv(

    SYNTHETIC_DIR /

    "SAFE_MFI_Synthetic_Dataset_v1.csv"

)

print(dataset.shape)

dataset.head()

(10000, 40)


,customer_id,branch_id,application_date,region,rural_urban,age,gender,marital_status,household_size,education_level,...,model_version,decision_timestamp,monitoring_date,risk_score,prediction_probability,risk_category,human_review,manual_override,drift_score,loan_status
0,CUST000001,BR008,2023-02-21,East,Urban,32,Male,Married,2,NaN,...,SAFE-MFI-v1,2025-05-09,2024-03-27,6,0.9526,Low,No,No,0.224,1
1,CUST000002,BR018,2023-01-14,South,Rural,46,Female,Divorced,3,Primary,...,SAFE-MFI-v1,2023-09-13,2025-02-14,2,0.7311,Medium,No,No,0.296,1
2,CUST000003,BR019,2025-01-10,Central,Urban,46,Male,Single,3,Primary,...,SAFE-MFI-v1,2024-06-30,2023-05-27,6,0.9526,Low,No,No,0.405,1
3,CUST000004,BR011,2025-12-29,South,Urban,38,Female,Widowed,4,Secondary,...,SAFE-MFI-v1,2024-10-06,2024-03-11,4,0.8808,Medium,No,No,0.116,0
4,CUST000005,BR026,2025-07-28,South,Rural,23,Male,Married,1,Secondary,...,SAFE-MFI-v1,2023-11-24,2025-05-14,5,0.9241,Low,No,No,0.142,1


In [21]:
##chech for missing data##
missing = dataset.isnull().sum()

missing

customer_id                   0
branch_id                     0
application_date              0
region                        0
rural_urban                   0
age                           0
gender                        0
marital_status                0
household_size                0
education_level            2545
employment_status             0
business_type                 0
business_age_years            0
monthly_income                0
monthly_expenses              0
savings_balance               0
existing_debt                 0
debt_to_income_ratio          0
savings_to_income_ratio       0
income_stability              0
collateral                    0
credit_history_years          0
previous_defaults             0
mobile_money_score            0
loan_amount                   0
loan_term_months              0
interest_rate                 0
repayment_history             0
loan_cycle                    0
group_lending                 0
model_version                 0
decision

In [22]:
##check for duplicates##
duplicates = dataset.duplicated().sum()

print(

    f"Duplicate Records: {duplicates}"

)

dataset = dataset.drop_duplicates()

Duplicate Records: 0


In [23]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              10000 non-null  object 
 1   branch_id                10000 non-null  object 
 2   application_date         10000 non-null  object 
 3   region                   10000 non-null  object 
 4   rural_urban              10000 non-null  object 
 5   age                      10000 non-null  int64  
 6   gender                   10000 non-null  object 
 7   marital_status           10000 non-null  object 
 8   household_size           10000 non-null  int64  
 9   education_level          7455 non-null   object 
 10  employment_status        10000 non-null  object 
 11  business_type            10000 non-null  object 
 12  business_age_years       10000 non-null  int64  
 13  monthly_income           10000 non-null  float64
 14  monthly_expenses       

In [7]:
##Descriptive statistics##
dataset.describe()

,age,household_size,business_age_years,monthly_income,monthly_expenses,savings_balance,existing_debt,debt_to_income_ratio,savings_to_income_ratio,credit_history_years,previous_defaults,mobile_money_score,loan_amount,loan_term_months,interest_rate,loan_cycle,risk_score,prediction_probability,drift_score,loan_status
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,37.557400,4.013600,10.028600,99911.591081,57601.060649,31418.563310,13260.750394,0.133134,0.314411,10.06250,0.309100,71.160690,9.960524e+04,19.111200,18.001520,3.516600,3.948600,0.825104,0.198827,0.824300
std,9.779901,1.956834,6.047019,48001.694935,31537.779347,26778.694975,11935.301652,0.093316,0.198515,6.03747,0.569025,16.134201,8.470079e+04,10.295195,5.708784,1.699886,2.492705,0.180518,0.120306,0.380584
min,18.000000,1.000000,0.000000,20000.000000,7125.890000,494.280000,25.240000,0.001000,0.008000,0.00000,0.000000,9.200000,9.964000e+02,6.000000,8.000000,1.000000,-7.000000,0.029300,0.001000,0.000000
25%,31.000000,3.000000,5.000000,66635.127500,35728.370000,13781.807500,5299.212500,0.064000,0.171000,5.00000,0.000000,60.900000,4.328663e+04,12.000000,13.100000,2.000000,3.000000,0.817600,0.107000,1.000000
50%,37.000000,4.000000,10.000000,89988.225000,50595.565000,23741.000000,9854.780000,0.112000,0.273000,10.00000,0.000000,73.300000,7.643354e+04,18.000000,18.000000,4.000000,4.000000,0.880800,0.178000,1.000000
75%,44.000000,5.000000,15.000000,121898.707500,71762.607500,41187.727500,17317.730000,0.178000,0.414000,15.00000,1.000000,83.800000,1.287380e+05,24.000000,22.930000,5.000000,6.000000,0.952600,0.270000,1.000000
max,70.000000,13.000000,20.000000,473921.030000,337271.860000,370503.030000,135706.100000,0.801000,1.943000,20.00000,2.000000,99.600000,1.280527e+06,36.000000,28.000000,6.000000,11.000000,0.995900,0.759000,1.000000


In [8]:
dataset.describe(include="object")

,customer_id,branch_id,application_date,region,rural_urban,gender,marital_status,education_level,employment_status,business_type,income_stability,collateral,repayment_history,group_lending,model_version,decision_timestamp,monitoring_date,risk_category,human_review,manual_override
count,10000,10000,10000,10000,10000,10000,10000,7455,10000,10000,10000,10000,10000,10000,10000,10000,10000,10000,10000,10000
unique,10000,50,1096,5,2,2,4,3,5,5,3,2,3,2,1,1096,1095,3,2,2
top,CUST009984,BR050,2025-06-06,South,Urban,Female,Divorced,Secondary,Salaried,Agriculture,Stable,No,Good,Yes,SAFE-MFI-v1,2024-03-15,2025-03-26,Low,No,No
freq,1,236,24,2044,5064,5087,2537,2510,2038,2046,3378,5452,5502,6523,10000,21,22,4670,8315,9506


In [24]:
##Outlier Detection##
numeric_columns = dataset.select_dtypes(

    include=np.number

).columns

numeric_columns

Index(['age', 'household_size', 'business_age_years', 'monthly_income',
       'monthly_expenses', 'savings_balance', 'existing_debt',
       'debt_to_income_ratio', 'savings_to_income_ratio',
       'credit_history_years', 'previous_defaults', 'mobile_money_score',
       'loan_amount', 'loan_term_months', 'interest_rate', 'loan_cycle',
       'risk_score', 'prediction_probability', 'drift_score', 'loan_status'],
      dtype='object')

In [25]:
outliers = {}

for col in numeric_columns:

    Q1 = dataset[col].quantile(0.25)

    Q3 = dataset[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR

    upper = Q3 + 1.5 * IQR

    count = (

        (dataset[col] < lower)

        |

        (dataset[col] > upper)

    ).sum()

    outliers[col] = count

outlier_df = pd.DataFrame({

    "Feature":outliers.keys(),

    "Outliers":outliers.values()

})

outlier_df

,Feature,Outliers
0,age,42
1,household_size,203
2,business_age_years,0
3,monthly_income,344
4,monthly_expenses,360
5,savings_balance,472
6,existing_debt,542
7,debt_to_income_ratio,325
8,savings_to_income_ratio,293
9,credit_history_years,0


In [11]:
outlier_df.to_csv(

    OUTPUT_DIR /

    "outlier_report.csv",

    index=False

)

In [28]:
##Convert dates##

date_columns = [

    "application_date",

    "decision_timestamp",

    "monitoring_date"

]

for col in date_columns:

    dataset[col] = pd.to_datetime(

        dataset[col]

    )


In [29]:
##Create additional features.##

dataset["expense_ratio"] = (

    dataset["monthly_expenses"]

    /

    dataset["monthly_income"]

)

dataset["loan_income_ratio"] = (

    dataset["loan_amount"]

    /

    dataset["monthly_income"]

)

dataset["savings_debt_ratio"] = (

    dataset["savings_balance"]

    /

    (

        dataset["existing_debt"]

        +

        1

    )

)

In [31]:
##Encode Categorical Variables##
categorical_columns = dataset.select_dtypes(

    include="object"

).columns

categorical_columns

Index(['customer_id', 'branch_id', 'region', 'rural_urban', 'gender',
       'marital_status', 'education_level', 'employment_status',
       'business_type', 'income_stability', 'collateral', 'repayment_history',
       'group_lending', 'model_version', 'risk_category', 'human_review',
       'manual_override'],
      dtype='object')

In [32]:
encoders = {}

for column in categorical_columns:

    encoder = LabelEncoder()

    dataset[column] = encoder.fit_transform(

        dataset[column].astype(str)

    )

    encoders[column] = encoder

dataset.head()

,customer_id,branch_id,application_date,region,rural_urban,age,gender,marital_status,household_size,education_level,...,risk_score,prediction_probability,risk_category,human_review,manual_override,drift_score,loan_status,expense_ratio,loan_income_ratio,savings_debt_ratio
0,0,7,2023-02-21,1,1,32,1,1,2,3,...,6,0.9526,1,0,0,0.224,1,0.486909,0.196121,2.471725
1,1,17,2023-01-14,3,0,46,0,0,3,0,...,2,0.7311,2,0,0,0.296,1,0.548069,0.381161,2.006313
2,2,18,2025-01-10,0,1,46,1,2,3,0,...,6,0.9526,1,0,0,0.405,1,0.594213,1.256579,0.849094
3,3,10,2025-12-29,3,1,38,0,3,4,1,...,4,0.8808,2,0,0,0.116,0,0.602575,2.152871,1.560450
4,4,25,2025-07-28,3,0,23,1,1,1,1,...,5,0.9241,1,0,0,0.142,1,0.493102,0.655693,4.804297


In [33]:
##Normalize Numerical Features##
scaler = StandardScaler()

numeric_columns = dataset.select_dtypes(

    include=np.number

).columns

numeric_columns = [

    c for c in numeric_columns

    if c != "loan_status"

]

dataset[numeric_columns] = scaler.fit_transform(

    dataset[numeric_columns]

)

In [34]:
##Train/Test Split##
X = dataset.drop(

    columns=[

        "loan_status"

    ]

)

y = dataset["loan_status"]

In [35]:
X_train,X_test,y_train,y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [36]:
print(X_train.shape)

print(X_test.shape)

print(y_train.shape)

print(y_test.shape)

(8000, 42)
(2000, 42)
(8000,)
(2000,)


In [39]:
##Export Processed Data##
train = X_train.copy()

train["loan_status"] = y_train

test = X_test.copy()

test["loan_status"] = y_test

train.to_csv(

    PROCESSED_DIR /

    "train.csv",

    index=False

)

test.to_csv(

    PROCESSED_DIR /

    "test.csv",

    index=False

)

dataset.to_csv(

    PROCESSED_DIR /

    "SAFE_MFI_processed.csv",

    index=False

)

print("="*70)

print("PREPROCESSING COMPLETE")

print("="*70)

print("Files Generated")

print("------------------------")

print("SAFE_MFI_processed.csv")

print("train.csv")

print("test.csv")

print("outlier_report.csv")

print()

print("Notebook 02 Complete")

PREPROCESSING COMPLETE
Files Generated
------------------------
SAFE_MFI_processed.csv
train.csv
test.csv
outlier_report.csv

Notebook 02 Complete
